# 02 - Data Cleaning and Preprocessing
### Student Course Completion Prediction Dataset

Picks up the issues logged in `reports/quality_report.csv` and resolves each one, then builds the
derived features that the EDA and dashboard run on. Output: `data/processed/clean_student_data.csv`.

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. Load raw data

Same loading logic as notebook 01: full local CSV first, then Kaggle, then the sample.

In [2]:
def load_dataset():
    local_full_path = "../data/raw/Course_Completion_Prediction.csv"
    if os.path.exists(local_full_path):
        df = pd.read_csv(local_full_path)
        print(f"Loaded full dataset from local file: {df.shape[0]} rows, {df.shape[1]} columns")
        return df

    try:
        import kagglehub
        path = kagglehub.dataset_download(
            "nisargpatel344/student-course-completion-prediction-dataset"
        )
        csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
        df = pd.read_csv(os.path.join(path, csv_files[0]))
        print(f"Loaded full dataset from Kaggle: {df.shape[0]} rows, {df.shape[1]} columns")
        return df
    except Exception as e:
        print(f"Kaggle download not available here ({type(e).__name__}: {e})")
        sample_path = "../data/raw/sample_200.csv"
        df = pd.read_csv(sample_path)
        print(f"Loaded sample dataset: {df.shape[0]} rows, {df.shape[1]} columns")
        return df

df = load_dataset()
df.head()

Loaded full dataset from local file: 100000 rows, 40 columns


,Student_ID,Name,Gender,Age,Education_Level,Employment_Status,City,Device_Type,Internet_Connection_Quality,Course_ID,Course_Name,Category,Course_Level,Course_Duration_Days,Instructor_Rating,Login_Frequency,Average_Session_Duration_Min,Video_Completion_Rate,Discussion_Participation,Time_Spent_Hours,Days_Since_Last_Login,Notifications_Checked,Peer_Interaction_Score,Assignments_Submitted,Assignments_Missed,Quiz_Attempts,Quiz_Score_Avg,Project_Grade,Progress_Percentage,Rewatch_Count,Enrollment_Date,Payment_Mode,Fee_Paid,Discount_Used,Payment_Amount,App_Usage_Percentage,Reminder_Emails_Clicked,Support_Tickets_Raised,Satisfaction_Rating,Completed
0,STU100000,Vihaan Patel,Male,19,Diploma,Student,Indore,Laptop,Medium,C102,Data Analysis with Python,Programming,Intermediate,60,4.7,3,30,55.0,2,0.5,1,6,4.3,8,1,5,80.9,71.2,70.8,0,01-06-2024,Scholarship,No,No,1740,49,3,4,3.5,Completed
1,STU100001,Arjun Nair,Female,17,Bachelor,Student,Delhi,Laptop,Low,C106,Machine Learning A-Z,Programming,Advanced,90,4.6,4,37,84.1,2,0.9,3,5,7.8,4,6,3,78.4,42.5,55.6,2,27-04-2025,Credit Card,Yes,No,6147,86,0,0,4.5,Not Completed
2,STU100002,Aditya Bhardwaj,Female,34,Master,Student,Chennai,Mobile,Medium,C101,Python Basics,Programming,Beginner,45,4.6,5,9,75.6,3,0.5,19,5,6.7,8,2,3,100.0,87.9,78.8,2,20-01-2024,NetBanking,Yes,No,4280,85,1,0,5.0,Completed
3,STU100003,Krishna Singh,Female,29,Diploma,Employed,Surat,Mobile,High,C105,UI/UX Design Fundamentals,Design,Beginner,40,4.4,2,27,63.3,1,7.4,19,9,6.4,0,10,4,59.1,51.4,24.7,4,13-05-2025,UPI,Yes,No,3812,42,2,3,3.8,Completed
4,STU100004,Krishna Nair,Female,19,Master,Self-Employed,Lucknow,Laptop,Medium,C106,Machine Learning A-Z,Programming,Advanced,90,4.6,2,36,86.4,1,0.5,4,7,7.5,5,5,8,84.8,93.0,64.9,4,19-12-2024,Debit Card,Yes,Yes,5486,91,3,0,4.0,Completed


In [3]:
quality_report = pd.read_csv("../reports/quality_report.csv")
print(f"{len(quality_report)} issues logged in the quality report, resolving each below.")
quality_report

38 issues logged in the quality report, resolving each below.


,column,issue_type,row_count,pct_of_data,planned_resolution
0,Age,implausible value (<10 or >90),0,0.00,cap or flag for manual review in notebook 02
1,Video_Completion_Rate,out of 0-100 range,0,0.00,"cap to [0, 100] in notebook 02"
2,Progress_Percentage,out of 0-100 range,0,0.00,"cap to [0, 100] in notebook 02"
3,App_Usage_Percentage,out of 0-100 range,0,0.00,"cap to [0, 100] in notebook 02"
4,Quiz_Score_Avg,out of 0-100 range,0,0.00,"cap to [0, 100] in notebook 02"
5,Project_Grade,out of 0-100 range,0,0.00,"cap to [0, 100] in notebook 02"
6,Instructor_Rating,out of 1-5 range,0,0.00,flag for manual review
7,Login_Frequency,negative value,0,0.00,cap to 0 or investigate in notebook 02
8,Time_Spent_Hours,negative value,0,0.00,cap to 0 or investigate in notebook 02
9,Course_Duration_Days,negative value,0,0.00,cap to 0 or investigate in notebook 02


## 2. Drop personally identifiable information

`Name` plays no analytical role and is a privacy concern. Dropped before anything else happens.

In [4]:
df = df.drop(columns=["Name"])
print("Dropped: Name")
print("Remaining columns:", df.shape[1])

Dropped: Name
Remaining columns: 39


## 3. Parse Enrollment_Date

Stored as text in `dd-mm-yyyy` format. Converted to a real datetime, from which `enrollment_month`
and `enrollment_year` get derived for trend views later.

In [5]:
df["Enrollment_Date"] = pd.to_datetime(df["Enrollment_Date"], format="%d-%m-%Y", errors="coerce")

unparsed = df["Enrollment_Date"].isnull().sum()
print(f"Rows that failed to parse: {unparsed}")

df["enrollment_year"] = df["Enrollment_Date"].dt.year
df["enrollment_month"] = df["Enrollment_Date"].dt.to_period("M").astype(str)

print(f"Enrollment date range: {df['Enrollment_Date'].min()} to {df['Enrollment_Date'].max()}")
df[["Enrollment_Date", "enrollment_year", "enrollment_month"]].head()

Rows that failed to parse: 0
Enrollment date range: 2023-10-17 00:00:00 to 2025-10-06 00:00:00


,Enrollment_Date,enrollment_year,enrollment_month
0,2024-06-01,2024,2024-06
1,2025-04-27,2025,2025-04
2,2024-01-20,2024,2024-01
3,2025-05-13,2025,2025-05
4,2024-12-19,2024,2024-12


## 4. Document the Scholarship / Fee_Paid quirk

`Fee_Paid = No` but `Payment_Amount > 0` shows up for a meaningful share of rows, every one of them
tagged `Payment_Mode = Scholarship`. This isn't corrupted data, it just means scholarship students
still had a reduced or administrative amount recorded. Rather than overwrite it, it gets captured
as an explicit flag so it's visible and intentional in every downstream analysis.

In [6]:
df["is_scholarship_partial_pay"] = (
    (df["Fee_Paid"] == "No") & (df["Payment_Amount"] > 0)
).astype(int)

print(f"Flagged rows: {df['is_scholarship_partial_pay'].sum()} "
        f"({df['is_scholarship_partial_pay'].mean()*100:.2f}% of the data)")

# Confirm it is fully explained by Scholarship payment mode
flagged = df[df["is_scholarship_partial_pay"] == 1]
print("\nPayment_Mode breakdown for flagged rows:")
print(flagged["Payment_Mode"].value_counts())

Flagged rows: 10039 (10.04% of the data)

Payment_Mode breakdown for flagged rows:
Payment_Mode
Scholarship    10039
Name: count, dtype: int64


## 5. Note on Progress_Percentage vs Completed

The quality report flagged that a meaningful share of `Completed` students have under 50% logged
progress. This is not something to "fix" in cleaning, since both fields look valid on their own,
it's a genuine pattern that gets investigated properly in the EDA notebook (03), where the
hypothesis is that `Quiz_Score_Avg` and `Project_Grade` are the real gatekeepers of completion
rather than raw progress tracking. No change made here, just carrying both columns forward as-is.

## 6. Standardize categorical formatting

Defensive pass, even though the quality report found the categories already well formatted.

In [7]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()
for col in ["Student_ID"]:
    if col in categorical_cols:
        categorical_cols.remove(col)

for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

print("Standardized whitespace on:", categorical_cols)

Standardized whitespace on: ['Gender', 'Education_Level', 'Employment_Status', 'City', 'Device_Type', 'Internet_Connection_Quality', 'Course_ID', 'Course_Name', 'Category', 'Course_Level', 'Payment_Mode', 'Fee_Paid', 'Discount_Used', 'Completed', 'enrollment_month']


## 7. Outlier review

The quality report flagged IQR-based outliers scattered across most numeric columns, each at a
small percentage (well under 10%). Rather than blanket-capping everything, each field gets a quick
judgment call: cap it if the value looks like a plausible data entry slip, leave it if it looks
like real (if extreme) student behavior.

For a dataset like this, extreme-but-plausible values (a very engaged student with unusually high
`Discussion_Participation`, or a student with a long `Days_Since_Last_Login` because they simply
stopped attending) are meaningful signal, not noise, especially for a completion analysis. Capping
those away would quietly destroy some of the most useful rows in the dataset. So here, capping is
applied narrowly, only to fields where a value sits outside what is physically or structurally
possible, everything else is left untouched and just noted.

In [8]:
def cap_outliers_iqr(series, factor=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - factor * iqr, q3 + factor * iqr
    return series.clip(lower=lower, upper=upper)

# Fields where physically impossible values would be a real data error worth capping
hard_cap_fields = {
    "Video_Completion_Rate": (0, 100),
    "Progress_Percentage": (0, 100),
    "App_Usage_Percentage": (0, 100),
    "Quiz_Score_Avg": (0, 100),
    "Project_Grade": (0, 100),
}

for col, (lo, hi) in hard_cap_fields.items():
    before = ((df[col] < lo) | (df[col] > hi)).sum()
    df[col] = df[col].clip(lower=lo, upper=hi)
    print(f"{col}: {before} rows outside [{lo}, {hi}] hard-capped (0 expected here, confirms notebook 01 findings)")

print("\nAll other IQR-flagged outliers (engagement and performance fields) are left as-is,")
print("treated as real variation in student behavior rather than data entry errors.")

Video_Completion_Rate: 0 rows outside [0, 100] hard-capped (0 expected here, confirms notebook 01 findings)
Progress_Percentage: 0 rows outside [0, 100] hard-capped (0 expected here, confirms notebook 01 findings)
App_Usage_Percentage: 0 rows outside [0, 100] hard-capped (0 expected here, confirms notebook 01 findings)
Quiz_Score_Avg: 0 rows outside [0, 100] hard-capped (0 expected here, confirms notebook 01 findings)
Project_Grade: 0 rows outside [0, 100] hard-capped (0 expected here, confirms notebook 01 findings)

All other IQR-flagged outliers (engagement and performance fields) are left as-is,
treated as real variation in student behavior rather than data entry errors.


## 8. Missing values

Confirmed zero missing values in notebook 01. Re-checking here as a safety net before export.

In [9]:
missing_after = df.isnull().sum()
missing_after = missing_after[missing_after > 0]
if len(missing_after) == 0:
    print("No missing values, nothing to impute.")
else:
    print("Missing values found, applying median (numeric) / mode (categorical) imputation:")
    for col in missing_after.index:
        if df[col].dtype in [np.float64, np.int64]:
            fill_value = df[col].median()
            df[col] = df[col].fillna(fill_value)
            print(f"  {col}: filled with median ({fill_value})")
        else:
            fill_value = df[col].mode()[0]
            df[col] = df[col].fillna(fill_value)
            print(f"  {col}: filled with mode ({fill_value})")

No missing values, nothing to impute.


## 9. Duplicates

Confirmed zero in notebook 01, dropping defensively in case this runs on a different data pull.

In [10]:
before = len(df)
df = df.drop_duplicates()
if "Student_ID" in df.columns:
    df = df.drop_duplicates(subset=["Student_ID"], keep="first")
after = len(df)
print(f"Rows dropped as duplicates: {before - after}")
print(f"Final row count: {after}")

Rows dropped as duplicates: 0
Final row count: 100000


## 10. Feature engineering

Building the derived fields the EDA and dashboard will actually use.

In [11]:
# completed_flag: clean 0/1 version of the target, keeping the text column too for readable charts
df["completed_flag"] = (df["Completed"] == "Completed").astype(int)

# performance_score: average of quiz and project performance
df["performance_score"] = (df["Quiz_Score_Avg"] + df["Project_Grade"]) / 2

# assignment_completion_rate
total_assignments = df["Assignments_Submitted"] + df["Assignments_Missed"]
df["assignment_completion_rate"] = np.where(
    total_assignments > 0,
    df["Assignments_Submitted"] / total_assignments,
    np.nan
)

# engagement_score: scaled combination of login frequency, time spent, video completion, app usage
from sklearn.preprocessing import MinMaxScaler

engagement_inputs = ["Login_Frequency", "Time_Spent_Hours", "Video_Completion_Rate", "App_Usage_Percentage"]
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[engagement_inputs])
df["engagement_score"] = scaled.mean(axis=1) * 100  # 0-100 scale for readability

# engagement_tier: Low / Medium / High buckets off engagement_score
df["engagement_tier"] = pd.qcut(
    df["engagement_score"], q=3, labels=["Low", "Medium", "High"]
)

# age_band for cleaner demographic slicing
df["age_band"] = pd.cut(
    df["Age"],
    bins=[0, 20, 25, 30, 35, 100],
    labels=["Under 20", "20-24", "25-29", "30-34", "35+"]
)

print("New columns added:")
new_cols = ["completed_flag", "performance_score", "assignment_completion_rate",
            "engagement_score", "engagement_tier", "age_band",
            "is_scholarship_partial_pay", "enrollment_year", "enrollment_month"]
print(new_cols)
df[new_cols].head()

New columns added:
['completed_flag', 'performance_score', 'assignment_completion_rate', 'engagement_score', 'engagement_tier', 'age_band', 'is_scholarship_partial_pay', 'enrollment_year', 'enrollment_month']


,completed_flag,performance_score,assignment_completion_rate,engagement_score,engagement_tier,age_band,is_scholarship_partial_pay,enrollment_year,enrollment_month
0,1,76.05,0.888889,30.421760,Low,Under 20,1,2024,2024-06
1,0,60.45,0.400000,49.402797,High,Under 20,0,2025,2025-04
2,1,93.95,0.800000,48.181858,High,30-34,0,2024,2024-01
3,1,55.25,0.000000,36.064115,Low,25-29,0,2025,2025-05
4,1,88.90,0.500000,47.526958,High,Under 20,0,2024,2024-12


In [12]:
print("Engagement tier distribution:")
print(df["engagement_tier"].value_counts())
print()
print("Age band distribution:")
print(df["age_band"].value_counts().sort_index())
print()
print("Completion rate overall:", round(df["completed_flag"].mean() * 100, 2), "%")

Engagement tier distribution:
engagement_tier
Low       33334
Medium    33333
High      33333
Name: count, dtype: int64

Age band distribution:
age_band
Under 20    20425
20-24       29578
25-29       29732
30-34       15470
35+          4795
Name: count, dtype: int64

Completion rate overall: 49.03 %


## 11. Final structure check

In [13]:
print("Final shape:", df.shape)
df.info()

Final shape: (100000, 48)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 48 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   Student_ID                    100000 non-null  object        
 1   Gender                        100000 non-null  object        
 2   Age                           100000 non-null  int64         
 3   Education_Level               100000 non-null  object        
 4   Employment_Status             100000 non-null  object        
 5   City                          100000 non-null  object        
 6   Device_Type                   100000 non-null  object        
 7   Internet_Connection_Quality   100000 non-null  object        
 8   Course_ID                     100000 non-null  object        
 9   Course_Name                   100000 non-null  object        
 10  Category                      100000 non-null  object  

## 12. Export cleaned dataset

In [14]:
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/clean_student_data.csv", index=False)
print(f"Saved {df.shape[0]} rows x {df.shape[1]} columns to ../data/processed/clean_student_data.csv")

Saved 100000 rows x 48 columns to ../data/processed/clean_student_data.csv


## Summary

Resolved in this notebook:
- dropped `Name` (PII)
- parsed `Enrollment_Date` to a real datetime, derived `enrollment_year` / `enrollment_month`
- documented the Scholarship payment quirk as an explicit `is_scholarship_partial_pay` flag instead of overwriting it
- confirmed zero missing values and zero duplicates, with imputation/dedup logic still included as a safety net
- hard-capped only the handful of fields that must sit within 0-100 (found nothing to actually cap, notebook 01 already confirmed this)
- left the wider IQR-flagged outliers untouched, treating them as real student behavior

New features for EDA and the dashboard:
- `completed_flag`, `performance_score`, `assignment_completion_rate`
- `engagement_score` and `engagement_tier` (Low / Medium / High)
- `age_band`

Next: `03_exploratory_data_analysis.ipynb` runs the full univariate, bivariate, and multivariate
analysis on `clean_student_data.csv`.